## Import libraries

In [1]:
# import os
# from pathlib import Path

import pandas as pd
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
from psycopg2.extras import execute_batch

## Functions to connect to db, create new db.
## Created new db 'sp500_rpoject'

In [19]:
DB_NAME = "sp500_project"
USER = "postgres"
HOST = "localhost"
PORT = "5432"
PWD_FILE = "pwd.txt"

with open(PWD_FILE, "r") as f:
    PWD = f.read().strip()

In [6]:
def connect(db_nm: str = 'postgres',
            pwd: str = PWD,
            usr_nm: str = USER, 
            host: str = HOST,
            port: str = PORT,
           ) -> psycopg2.extensions.connection | None:
    conn = None
    
    try:
            conn = psycopg2.connect(
            dbname=db_nm,
            user=usr_nm,
            password=pwd,
            host=host,
            port=port
            )
    except Exception as e:
        print(f'Failed to connect to database. Error: {str(e)}')
    else:
        print(f'Connected to database {db_nm}')
    finally:
        return conn

def create_database(db_name: str,
                    usr_nm: str = USER,
                    pwd: str = PWD,
                    host: str = HOST,
                    port: str = PORT) -> None:
    conn = psycopg2.connect(
        dbname='postgres',
        user=usr_nm,
        password=pwd,
        host=host,
        port=port
    )
    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    cur = conn.cursor()

    try:
        cur.execute(f"CREATE DATABASE {db_name};")
    except Exception as e:
        print(f"Failed to create database '{db_name}'. Error: {str(e)}")
    else:
        print(f"Database '{db_name}' created.")
    finally:
        cur.close()
        conn.close()

In [7]:
create_database(DB_NAME)

conn = connect(DB_NAME)
cur = conn.cursor()

cur.close()
conn.close()

Failed to create database 'sp500_project'. Error: database "sp500_project" already exists

Connected to database sp500_project


## Clean data before converting to SQL. Problem: null values for adj_close in stocks

In [36]:
stocks_clean = stocks.copy()
companies_clean = companies.copy()
index_clean = index_df.copy()

stocks_clean = stocks_clean.rename(columns={
    'Date': 'date',
    'Symbol': 'symbol',
    'Adj Close': 'adj_close',
    'Close': 'close',
    'High': 'high',
    'Low': 'low',
    'Open': 'open',
    'Volume': 'volume'
})

companies_clean = companies_clean.rename(columns={
    'Exchange': 'exchange',
    'Symbol': 'symbol',
    'Shortname': 'shortname',
    'Longname': 'longname',
    'Sector': 'sector',
    'Industry': 'industry',
    'Currentprice': 'current_price',
    'Marketcap': 'market_cap',
    'Ebitda': 'ebitda',
    'Revenuegrowth': 'revenue_growth',
    'City': 'city',
    'State': 'state',
    'Country': 'country',
    'Fulltimeemployees': 'fulltime_employees',
    'Longbusinesssummary': 'long_business_summary',
    'Weight': 'weight'
})

index_clean = index_clean.rename(columns={
    'Date': 'date',
    'S&P500': 'sp500'
})

stocks_clean['date'] = pd.to_datetime(stocks_clean['date'])
index_clean['date'] = pd.to_datetime(index_clean['date'])

stocks_clean = stocks_clean.dropna(subset=['adj_close']).copy()

stocks_clean = stocks_clean.sort_values(['symbol', 'date']).reset_index(drop=True)
companies_clean = companies_clean.sort_values('symbol').reset_index(drop=True)
index_clean = index_clean.sort_values('date').reset_index(drop=True)

stocks_clean['date'] = stocks_clean['date'].dt.date
index_clean['date'] = index_clean['date'].dt.date

## Converting to SQL

In [47]:
conn = connect(DB_NAME)
cur = conn.cursor()

cur.execute("""
DROP TABLE IF EXISTS stocks;
DROP TABLE IF EXISTS companies;
DROP TABLE IF EXISTS market_index;
""")

cur.execute("""
CREATE TABLE companies (
    exchange TEXT,
    symbol TEXT PRIMARY KEY,
    shortname TEXT,
    longname TEXT,
    sector TEXT,
    industry TEXT,
    current_price DOUBLE PRECISION,
    market_cap BIGINT,
    ebitda DOUBLE PRECISION,
    revenue_growth DOUBLE PRECISION,
    city TEXT,
    state TEXT,
    country TEXT,
    fulltime_employees DOUBLE PRECISION,
    long_business_summary TEXT,
    weight DOUBLE PRECISION
);
""")

cur.execute("""
CREATE TABLE market_index (
    date DATE PRIMARY KEY,
    sp500 DOUBLE PRECISION
);
""")

cur.execute("""
CREATE TABLE stocks (
    date DATE,
    symbol TEXT,
    adj_close DOUBLE PRECISION,
    close DOUBLE PRECISION,
    high DOUBLE PRECISION,
    low DOUBLE PRECISION,
    open DOUBLE PRECISION,
    volume DOUBLE PRECISION,
    PRIMARY KEY (symbol, date)
);
""")

print("Tables created.")


# Inserting values into tables
companies_rows = list(companies_clean.itertuples(index=False, name=None))

insert_companies = """
INSERT INTO companies (
    exchange, symbol, shortname, longname, sector, industry,
    current_price, market_cap, ebitda, revenue_growth,
    city, state, country, fulltime_employees,
    long_business_summary, weight
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
"""

execute_batch(cur, insert_companies, companies_rows, page_size=1000)
print("companies loaded.")

index_rows = list(index_clean.itertuples(index=False, name=None))

insert_index = """
INSERT INTO market_index (date, sp500)
VALUES (%s, %s);
"""

execute_batch(cur, insert_index, index_rows, page_size=1000)
conn.commit()
cur.close()
conn.close()

print("market_index loaded.")

def load_stocks_in_chunks(df, chunk_size=50000):
    conn = connect(DB_NAME)
    cur = conn.cursor()

    insert_stocks = """
    INSERT INTO stocks (
        date, symbol, adj_close, close, high, low, open, volume
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s);
    """

    total_rows = len(df)

    for start in range(0, total_rows, chunk_size):
        end = min(start + chunk_size, total_rows)
        chunk = df.iloc[start:end]

        rows = list(chunk.itertuples(index=False, name=None))
        execute_batch(cur, insert_stocks, rows, page_size=5000)
        conn.commit()

        print(f"Loaded rows {start} to {end} out of {total_rows}")

    cur.close()
    conn.close()
    print("stocks loaded.")

load_stocks_in_chunks(stocks_clean)

Connected to database sp500_project
Tables created.
companies loaded.
market_index loaded.
Connected to database sp500_project
Loaded rows 0 to 50000 out of 617831
Loaded rows 50000 to 100000 out of 617831
Loaded rows 100000 to 150000 out of 617831
Loaded rows 150000 to 200000 out of 617831
Loaded rows 200000 to 250000 out of 617831
Loaded rows 250000 to 300000 out of 617831
Loaded rows 300000 to 350000 out of 617831
Loaded rows 350000 to 400000 out of 617831
Loaded rows 400000 to 450000 out of 617831
Loaded rows 450000 to 500000 out of 617831
Loaded rows 500000 to 550000 out of 617831
Loaded rows 550000 to 600000 out of 617831
Loaded rows 600000 to 617831 out of 617831
stocks loaded.


In [48]:
conn = connect(DB_NAME)
cur = conn.cursor()

cur.execute('''
    SELECT * FROM stocks LIMIT 5
''')
print(cur.fetchall())

cur.close()
conn.close()

Connected to database sp500_project
[(datetime.date(2013, 1, 2), 'ABBV', 21.629180908203125, 35.119998931884766, 35.400001525878906, 34.099998474121094, 34.91999816894531, 13767900.0), (datetime.date(2013, 1, 3), 'ABBV', 21.450580596923828, 34.83000183105469, 35.0, 34.15999984741211, 35.0, 16739300.0), (datetime.date(2013, 1, 4), 'ABBV', 21.179594039916992, 34.38999938964844, 34.88999938964844, 34.25, 34.619998931884766, 21372100.0), (datetime.date(2013, 1, 7), 'ABBV', 21.222700119018555, 34.459999084472656, 35.45000076293945, 34.150001525878906, 34.150001525878906, 17897100.0), (datetime.date(2013, 1, 8), 'ABBV', 20.76080894470215, 33.709999084472656, 34.63999938964844, 33.36000061035156, 34.290000915527344, 17863300.0)]


In [49]:
conn = connect(DB_NAME)
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM stocks;")
print("stocks rows:", cur.fetchone()[0])

cur.execute("SELECT COUNT(*) FROM companies;")
print("companies rows:", cur.fetchone()[0])

cur.execute("SELECT COUNT(*) FROM market_index;")
print("market_index rows:", cur.fetchone()[0])

cur.close()
conn.close()

Connected to database sp500_project
stocks rows: 617831
companies rows: 502
market_index rows: 2517


## Step 1. Joins